In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 1. Path ke folder gambar train
BASE_DIR = r"D:\-\GEMASTIK 8"
IMG_DIR = os.path.join(BASE_DIR, r"data\csiro-biomass\train_images")

# Ambil 4 sampel gambar dari folder train
all_images = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_images = all_images[:4]  # Ambil 4 gambar pertama untuk diuji

print(f" Memproses segmentasi untuk {len(sample_images)} sampel gambar...")

# 2. Fungsi Ekstraksi Masker Segmentasi Vegetasi
def process_vegetation_segmentation(img_pil: Image.Image):
    img_np = np.array(img_pil, dtype=np.float32) / 255.0
    R, G, B = img_np[:, :, 0], img_np[:, :, 1], img_np[:, :, 2]
    
    # --- A. Excess Green Index (ExG) ---
    exg = 2.0 * G - R - B
    exg_mask = np.where(exg > 0.05, exg, 0.0)
    exg_mask = (exg_mask - exg_mask.min()) / (exg_mask.max() - exg_mask.min() + 1e-8)
    
    # --- B. Excess Red Index / Brown Mask (ExR) ---
    exr = 1.4 * R - G
    # Deteksi warna kecokelatan/kuning (Dead Vegetation)
    dead_mask = np.where((exr > 0.02) & (R > G * 0.85), exr, 0.0)
    dead_mask = (dead_mask - dead_mask.min()) / (dead_mask.max() - dead_mask.min() + 1e-8)
    
    # --- C. Combined Overlay Image ---
    # Background = Hitam, Green = Hijau, Dead = Cokelat/Merah
    overlay = np.zeros_like(img_np)
    overlay[:, :, 1] = exg_mask   # Green channel
    overlay[:, :, 0] = dead_mask  # Red channel
    
    return exg_mask, dead_mask, overlay

# 3. Visualisasi 4 Kolom Per Gambar
fig, axes = plt.subplots(len(sample_images), 4, figsize=(18, 4 * len(sample_images)))

for idx, img_name in enumerate(sample_images):
    img_path = os.path.join(IMG_DIR, img_name)
    img_pil = Image.open(img_path).convert('RGB')
    
    exg_mask, dead_mask, overlay = process_vegetation_segmentation(img_pil)
    
    # Panel 1: Gambar Asli RGB
    axes[idx, 0].imshow(img_pil)
    axes[idx, 0].set_title(f"Gambar Asli #{idx+1}\n{img_name}", fontsize=10, fontweight='bold')
    axes[idx, 0].axis('off')
    
    # Panel 2: Masker Vegetasi Hijau (Clover & Green)
    axes[idx, 1].imshow(exg_mask, cmap='YlGn')
    axes[idx, 1].set_title("ExG Mask (Green/Clover)", fontsize=10, fontweight='bold', color='green')
    axes[idx, 1].axis('off')
    
    # Panel 3: Masker Vegetasi Mati (Dead)
    axes[idx, 2].imshow(dead_mask, cmap='YlOrRd')
    axes[idx, 2].set_title("ExR Mask (Dead Biomass)", fontsize=10, fontweight='bold', color='darkred')
    axes[idx, 2].axis('off')
    
    # Panel 4: Combined Segmentation Overlay
    axes[idx, 3].imshow(overlay)
    axes[idx, 3].set_title("Combined Overlay (Segmented)", fontsize=10, fontweight='bold', color='purple')
    axes[idx, 3].axis('off')

plt.suptitle("Pengujian Segmentasi Citra Vegetasi pada Dataset CSIRO Biomass", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()